In [1]:
from labels import LABELS

label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

In [2]:

import json
from pathlib import Path

# Path to annotated JSONs
ANNOTATED_DIR = Path(r"D:\Y4 Research\datasets\ocr_json\ing_nut_set1\new\annotated\min_conf=40")

# Loop through all JSON files
for jf in ANNOTATED_DIR.glob("*.json"):
    with open(jf, encoding="utf-8") as f:
        data = json.load(f)

    # Check length consistency
    assert len(data["tokens"]) == len(data["bboxes"]) == len(data["labels"]), \
        f"{jf.name}: tokens/bboxes/labels length mismatch"

    # Check label validity
    for l in data["labels"]:
        assert l in LABELS, f"{jf.name}: invalid label '{l}'"

print("✅ All annotations valid")


✅ All annotations valid


Dataset Creation first_set

In [3]:
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image
import json
import os

class FoodLabelDataset(Dataset):
    def __init__(self, ann_dir, img_dir, processor, label2id):
        self.ann_dir = Path(ann_dir)
        self.img_dir = Path(img_dir)
        self.processor = processor
        self.label2id = label2id

        # All annotation JSONs
        self.files = sorted(self.ann_dir.glob("*.json"))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        ann_path = self.files[idx]

        # Load annotation
        with open(ann_path, encoding="utf-8") as f:
            ann = json.load(f)

        # Resolve image path (support png/jpg/jpeg)
        image_id = ann["image_id"]
        image_path = None
        for ext in [".png", ".jpg", ".jpeg"]:
            candidate = self.img_dir / f"{image_id}{ext}"
            if candidate.exists():
                image_path = candidate
                break

        if image_path is None:
            raise FileNotFoundError(f"Image not found for {image_id}")

        image = Image.open(image_path).convert("RGB")

        # Convert labels → ids
        label_ids = [self.label2id[l] for l in ann["labels"]]

        # LayoutLMv3 processor
        encoding = self.processor(
            image,
            ann["tokens"],
            boxes=ann["bboxes"],
            word_labels=label_ids,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        # Remove batch dimension (Trainer expects this)
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}

        return encoding

In [4]:
from transformers import LayoutLMv3Processor

ANN_DIR = r"D:\Y4 Research\datasets\ocr_json\ing_nut_set1\new\annotated\min_conf=40\Set1\OCRs"
IMG_DIR = r"D:\Y4 Research\datasets\ocr_json\ing_nut_set1\new\annotated\min_conf=40\Set1\Images"

processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=False
)

dataset = FoodLabelDataset(
    ann_dir=ANN_DIR,
    img_dir=IMG_DIR,
    processor=processor,
    label2id=label2id
)

In [5]:
sample = dataset[0]
for k, v in sample.items():
    print(k, v.shape)

input_ids torch.Size([512])
attention_mask torch.Size([512])
bbox torch.Size([512, 4])
labels torch.Size([512])
pixel_values torch.Size([3, 224, 224])
